# Random Forest RAT classification

In [1]:
import sys
sys.path.append("..")

from src.preprocessing import encode_values, convert_to_numeric
from src.utils_train_models import (load_preprocessed_dataset, 
                                    generate_outcome_training_per_z, 
                                    store_windowed_training_analysis)
from src.utils_eval import compute_statistics
from src.config import path_list
import pandas as pd
import numpy as np
import os

In [2]:
MODEL_NAME = "RandomForest"

#### Data preprocessing

In [ ]:
dataset = load_preprocessed_dataset()
 
print("Precomputed dataset:")
print(dataset)

##### Output: preprocessed dataset generation
```small_test
Reading the content of /Users/username/NDA-Lab-Project4-RAT-Classification-TL/data/outcome_preprocess/feature_dataset.csv

Precomputed dataset:
             id  run node_name           location    modem_name  mcc  country  \
0         17443    1   Mark-10    Zagreb, Croatia  Quectel_BG96  219  Croatia   
1         17558    1   Mark-10    Zagreb, Croatia  Quectel_EC21  219  Croatia   
2         17760    1   Mark-10    Zagreb, Croatia  Quectel_BG96  219  Croatia   
3         17768    1   Mark-10    Zagreb, Croatia  Quectel_EC21  219  Croatia   
4         18830    1   Mark-10    Zagreb, Croatia  Quectel_EC21  219  Croatia   
...         ...  ...       ...                ...           ...  ...      ...   
51060267  23995    1    Mark-6    Munich, Germany  Quectel_BG96  262  Germany   
51060268  24001    1    Mark-1  Würzburg, Germany  Quectel_BG96  262  Germany   
51060269  24003    1    Mark-6    Munich, Germany  Quectel_BG96  262  Germany   
51060270  24009    1    Mark-1  Würzburg, Germany  Quectel_BG96  262  Germany   
51060271  24011    1    Mark-6    Munich, Germany  Quectel_BG96  262  Germany   

         iso_code  rat rat_name  ...  tot_size_current  avg_speed_current  \
0              hr    0       2G  ...               0.0                0.0   
1              hr    0       2G  ...               0.0                0.0   
2              hr    0       2G  ...               0.0                0.0   
3              hr    0       2G  ...               0.0                0.0   
4              hr    0       2G  ...               0.0                0.0   
...           ...  ...      ...  ...               ...                ...   
51060267       de    9   NB-IoT  ...               0.0                0.0   
51060268       de    9   NB-IoT  ...               0.0                0.0   
51060269       de    9   NB-IoT  ...               0.0                0.0   
51060270       de    9   NB-IoT  ...               0.0                0.0   
51060271       de    9   NB-IoT  ...               0.0                0.0   

          tot_time_current  filesize_current direction_current  \
0                      0.0               0.0               0.0   
1                      0.0               0.0               0.0   
2                      0.0               0.0               0.0   
3                      0.0               0.0               0.0   
4                      0.0               0.0               0.0   
...                    ...               ...               ...   
51060267               0.0               0.0               0.0   
51060268               0.0               0.0               0.0   
51060269               0.0               0.0               0.0   
51060270               0.0               0.0               0.0   
51060271               0.0               0.0               0.0   

         timestamp_pw_idle  timetamp_ms_pw_idle  diff_pw_idle  \
0                      0.0                  0.0           0.0   
1                      0.0                  0.0           0.0   
2                      0.0                  0.0           0.0   
3                      0.0                  0.0           0.0   
4                      0.0                  0.0           0.0   
...                    ...                  ...           ...   
51060267               0.0                  0.0           0.0   
51060268               0.0                  0.0           0.0   
51060269               0.0                  0.0           0.0   
51060270               0.0                  0.0           0.0   
51060271               0.0                  0.0           0.0   

          current_pw_idle  voltage_pw_idle  
0                     0.0              0.0  
1                     0.0              0.0  
2                     0.0              0.0  
3                     0.0              0.0  
4                     0.0              0.0  
...                   ...              ...  
51060267              0.0              0.0  
51060268              0.0              0.0  
51060269              0.0              0.0  
51060270              0.0              0.0  
51060271              0.0              0.0  

[51060272 rows x 27 columns]
```

##### Recover the correct column data type

In [ ]:
dataset = convert_to_numeric(dataset)

list_non_numerical = []
list_numerical = []
for col in dataset.columns:
    if pd.api.types.is_numeric_dtype(dataset[col]):
        list_numerical.append(col)
    else:
        list_non_numerical.append(col)

print("Numerical column list:\n")
print(list_numerical)
print()
print("Non numerical column list:\n")
print(list_non_numerical)

##### Output:
```small_text
Numerical column list:

['id', 'run', 'mcc', 'iso_code', 'rat', 'timestamp_throughput', 'tot_size_throughput', 'avg_speed_throughput', 'tot_time_throughput', 'timestamp_current', 'tot_size_current', 'avg_speed_current', 'tot_time_current', 'filesize_current', 'direction_current', 'timestamp_pw_idle', 'timetamp_ms_pw_idle', 'diff_pw_idle', 'current_pw_idle', 'voltage_pw_idle']

Non numerical column list:

['node_name', 'location', 'modem_name', 'country', 'rat_name', 'filesize_throughput', 'direction_throughput']
```

##### RAT_NAME saved as labels

In [ ]:
RAT_NAME = [label for label in np.unique(dataset["rat_name"])]
print("RAT classification, labels name: {}".format(RAT_NAME))
dataset.drop("rat_name", axis=1, inplace=True)

##### Output:
```small_text
RAT classification, labels name: ['2G', '3G', 'LTE CAT1', 'LTE-M', 'NB-IoT']
```

#### Dataset encoding

In [ ]:
dataset = encode_values(dataset)
dataset.to_csv(path_list["ENCODED_DATASET"], index=False)
print("Labels encoding process has been successfully completed and stored\n")
print("Encoded dataset obtained\n")
print(dataset)

##### Output:
```small_text
Column list with meaningful values of type string

['node_name', 'location', 'modem_name', 'country', 'filesize_throughput', 'direction_throughput']
Encoding string objects within the feature dataset...

Outcome 1 step encoding

   node_name  encoded
0     Mark-1        0
1    Mark-10        1
2    Mark-11        2
3    Mark-12        3
4     Mark-2        4
5     Mark-3        5
6     Mark-4        6
7     Mark-5        7
8     Mark-6        8
9     Mark-7        9
10    Mark-9       10
Column: node_name, Number of rows with NaN value: 11868601

Outcome 2 step encoding

            location  encoded
0       Milan, Italy        0
1    Munich, Germany        1
2  Trondheim, Norway        2
3  Würzburg, Germany        3
4    Zagreb, Croatia        4
Column: location, Number of rows with NaN value: 15631980

Outcome 3 step encoding

     modem_name  encoded
0  Quectel_BG96        0
1  Quectel_EC21        1
Column: modem_name, Number of rows with NaN value: 16006451

Outcome 4 step encoding

   country  encoded
0  Croatia        0
1  Germany        1
2    Italy        2
3   Norway        3
Column: country, Number of rows with NaN value: 15824230

Outcome 5 step encoding

  filesize_throughput  encoded
0                 0.0        0
1               100KB        1
2               200KB        2
3                 2MB        3
4               500KB        4
5                50KB        5
6                 5MB        6
Column: filesize_throughput, Number of rows with NaN value: 24238

Outcome 6 step encoding

  direction_throughput  encoded
0                  0.0        0
1             Downlink        1
2               Uplink        2
Column: direction_throughput, Number of rows with NaN value: 24238

Labels encoding process has been successfully completed and stored

Encoded dataset obtained

             id  run  node_name  location  modem_name  mcc  country  iso_code  \
0         17443    1          1         4           0  219        0       0.0   
1         17558    1          1         4           1  219        0       0.0   
2         17760    1          1         4           0  219        0       0.0   
3         17768    1          1         4           1  219        0       0.0   
4         18830    1          1         4           1  219        0       0.0   
...         ...  ...        ...       ...         ...  ...      ...       ...   
51060267  23995    1          8         1           0  262        1       0.0   
51060268  24001    1          0         3           0  262        1       0.0   
51060269  24003    1          8         1           0  262        1       0.0   
51060270  24009    1          0         3           0  262        1       0.0   
51060271  24011    1          8         1           0  262        1       0.0   

          rat  timestamp_throughput  ...  tot_size_current  avg_speed_current  \
0           0                   0.0  ...               0.0                0.0   
1           0                   0.0  ...               0.0                0.0   
2           0                   0.0  ...               0.0                0.0   
3           0                   0.0  ...               0.0                0.0   
4           0                   0.0  ...               0.0                0.0   
...       ...                   ...  ...               ...                ...   
51060267    9                   0.0  ...               0.0                0.0   
51060268    9                   0.0  ...               0.0                0.0   
51060269    9                   0.0  ...               0.0                0.0   
51060270    9                   0.0  ...               0.0                0.0   
51060271    9                   0.0  ...               0.0                0.0   

          tot_time_current  filesize_current  direction_current  \
0                      0.0               0.0                0.0   
1                      0.0               0.0                0.0   
2                      0.0               0.0                0.0   
3                      0.0               0.0                0.0   
4                      0.0               0.0                0.0   
...                    ...               ...                ...   
51060267               0.0               0.0                0.0   
51060268               0.0               0.0                0.0   
51060269               0.0               0.0                0.0   
51060270               0.0               0.0                0.0   
51060271               0.0               0.0                0.0   

          timestamp_pw_idle  timetamp_ms_pw_idle  diff_pw_idle  \
0                       0.0                  0.0           0.0   
1                       0.0                  0.0           0.0   
2                       0.0                  0.0           0.0   
3                       0.0                  0.0           0.0   
4                       0.0                  0.0           0.0   
...                     ...                  ...           ...   
51060267                0.0                  0.0           0.0   
51060268                0.0                  0.0           0.0   
51060269                0.0                  0.0           0.0   
51060270                0.0                  0.0           0.0   
51060271                0.0                  0.0           0.0   

          current_pw_idle  voltage_pw_idle  
0                     0.0              0.0  
1                     0.0              0.0  
2                     0.0              0.0  
3                     0.0              0.0  
4                     0.0              0.0  
...                   ...              ...  
51060267              0.0              0.0  
51060268              0.0              0.0  
51060269              0.0              0.0  
51060270              0.0              0.0  
51060271              0.0              0.0  

[51060272 rows x 26 columns]
```

#### Generation of the windowed features dataset sorted by timestamp


In [ ]:
outcome_training_per_z = generate_outcome_training_per_z(dataset, RAT_NAME, "RandomForest")

##### Output:
```small_text
Measurement columns (8): ['tot_size_throughput', 'avg_speed_throughput', 'tot_time_throughput', 'filesize_throughput', 'direction_throughput', 'tot_size_current', 'avg_speed_current', 'tot_time_current']
Dataset was sorted by timestamps: ['timestamp_throughput', 'timestamp_current', 'timestamp_pw_idle']

Number of encoded groups: 85

Generation of the windowed dataset was ultimated
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 1000
25718000it [02:49, 152050.78it/s]                              9.65it/s]
25343000it [02:54, 145435.08it/s]                              2.04it/s]
Windowed dataset computed for z = 1000

Length dataset: 51381

Training a RandomForest classifier...
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 5000
25720000it [00:40, 637053.36it/s]                              5.13it/s]
25345000it [00:40, 628594.45it/s]                              5.02it/s]
Windowed dataset computed for z = 5000

Length dataset: 10453

Training a RandomForest classifier...
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 10000
25720000it [00:26, 969907.68it/s]                              2.02it/s] 
25350000it [00:25, 1007931.04it/s]                              2.57it/s]
Windowed dataset computed for z = 10000

Length dataset: 5347

Training a RandomForest classifier...
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 50000
25750000it [00:12, 2020728.97it/s]                              0.80it/s]
25350000it [00:14, 1761473.43it/s]                              6.76it/s]
Windowed dataset computed for z = 50000

Length dataset: 1261

Training a RandomForest classifier...
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 100000
25800000it [00:11, 2232253.93it/s]                              4.84it/s]
25400000it [00:11, 2226477.47it/s]                              6.81it/s]
Windowed dataset computed for z = 100000

Length dataset: 751

Training a RandomForest classifier...
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 500000
26000000it [00:10, 2385956.32it/s]                              4.87it/s]
25500000it [00:10, 2421680.72it/s]                              3.40it/s]
Windowed dataset computed for z = 500000

Length dataset: 337

Training a RandomForest classifier...
```

#### Statistics computation

In [ ]:
accuracy_per_z = compute_statistics(outcome_training_per_z, RAT_NAME)

##### Output:
```small_text
------------------------------------

Results for Z: 1000

Training time[s]: 0.2295970916748047

Accuracy: 0.8474079286887168

Global precision: 0.90641846844163

Global recall: 0.8474079286887168

Global f1score: 0.8571220178022118

------------------------------------

------------------------------------

Results for Z: 5000

Training time[s]: 0.14336490631103516

Accuracy: 0.8700229709035222

Global precision: 0.9280573275486926

Global recall: 0.8700229709035222

Global f1score: 0.8827328601937099

------------------------------------

------------------------------------

Results for Z: 10000

Training time[s]: 0.10723018646240234

Accuracy: 0.8304832713754647

Global precision: 0.8854884944806779

Global recall: 0.8304832713754647

Global f1score: 0.8320770564844951

------------------------------------

------------------------------------

Results for Z: 50000

Training time[s]: 0.1068427562713623

Accuracy: 0.770392749244713

Global precision: 0.8190966755143251

Global recall: 0.770392749244713

Global f1score: 0.7673581935476608

------------------------------------

------------------------------------

Results for Z: 100000

Training time[s]: 0.09945392608642578

Accuracy: 0.706601466992665

Global precision: 0.7749690085494476

Global recall: 0.706601466992665

Global f1score: 0.6933772851839688

------------------------------------

------------------------------------

Results for Z: 500000

Training time[s]: 0.09308433532714844

Accuracy: 0.698019801980198

Global precision: 0.7023222578121352

Global recall: 0.698019801980198

Global f1score: 0.6993471022684262

------------------------------------
```

In [29]:
max_accuracy = max(accuracy_per_z)
outcome_windowed_dataset_eval = {}
for index_row, accuracy in enumerate(accuracy_per_z):
    if accuracy == max_accuracy:
        outcome_windowed_dataset_eval["model_name"] = MODEL_NAME
        outcome_windowed_dataset_eval["window_size"] = outcome_training_per_z.loc[index_row]["z_value"]
        outcome_windowed_dataset_eval["selected_model"] = outcome_training_per_z.loc[index_row]["pretrained_model"]
        outcome_windowed_dataset_eval["windowed_dataset"] = outcome_training_per_z.loc[index_row]["features_set"]
        break
    
store_windowed_training_analysis(outcome_windowed_dataset_eval, MODEL_NAME)